# Multi-Crop Plant Disease Detection and Severity Estimation (Offline)
**AI/ML Course Project, Milestone 2 demo notebook**

Pipeline: PlantSeg data, crop selection, severity from lesion masks, multi-task MobileNetV3, evaluation, offline export, offline inference.

**How to use:** Runtime > Change runtime type > GPU (T4). Then Runtime > Run all. The first run downloads about 1.7 GB, so do it well before the viva and keep the outputs saved.

## 1. Setup and settings

In [ ]:
import os, glob, json, time, random, subprocess, zipfile, re
from pathlib import Path
from collections import defaultdict
from IPython.display import display
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

# ---------- project settings (change here) ----------
N_CROPS = 6            # how many crops to keep (those with the most images)
MIN_PER_CLASS = 60     # minimum official-training images per class
IMG_SIZE = 224
BATCH = 32
EPOCHS = 10
SEV_T1, SEV_T2 = 0.05, 0.20   # severity thresholds on lesion fraction: mild < T1 <= moderate < T2 <= severe
NUM_WORKERS = 2
PRETRAINED = True
DATA_DIR = Path("/content/data")
ZIP_PATH = Path("/content/Plantseg.zip")
META_CSV = None  # optionally set the exact CSV path
COLUMN_OVERRIDES = {}  # e.g. {"plant": "plant_type", "image": "image_name"}

## 2. Download PlantSeg
Source: Zenodo record 16665770 (version 6, `Plantseg.zip`, about 1.7 GB). If the download link fails, download the zip manually from https://zenodo.org/records/16665770, upload it to Google Drive, mount Drive, and unzip into `/content/data`.

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
if not any(DATA_DIR.rglob('*.csv')):
    if not ZIP_PATH.exists():
        subprocess.run(['wget', '--progress=bar:force', '-O', str(ZIP_PATH),
                        'https://zenodo.org/records/16665770/files/Plantseg.zip?download=1'], check=True)
    if not zipfile.is_zipfile(ZIP_PATH):
        raise ValueError('Invalid ZIP. Remove the failed download and retry or use Drive.')
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(DATA_DIR)
print('Dataset directory:', DATA_DIR)
# Drive fallback: mount Drive, then set ZIP_PATH to the uploaded Plantseg.zip and rerun.
# from google.colab import drive
# drive.mount('/content/drive')
# ZIP_PATH = Path('/content/drive/MyDrive/Plantseg.zip')

## 3. Load the metadata

In [ ]:
csvs = sorted(DATA_DIR.rglob('*.csv'))
print('CSV candidates:', [str(p) for p in csvs])
preferred = [p for p in csvs if 'meta' in p.name.lower()]
choices = preferred or csvs
if META_CSV is None and len(choices) != 1:
    raise ValueError('Set META_CSV in section 1 to the correct metadata CSV.')
meta_path = Path(META_CSV) if META_CSV else choices[0]
df = pd.read_csv(meta_path)
print(meta_path, df.columns.tolist(), 'rows:', len(df))
display(df.head())

The next cell finds the column names automatically. Check the printed choices and override them by hand if any is wrong.

In [ ]:
def norm(s):
    return re.sub(r'[^a-z0-9]', '', str(s).lower())

aliases = {
    'plant': ['plant_type', 'plant', 'crop', 'species', 'host', 'plant_name'],
    'disease': ['disease_type', 'disease', 'disease_name', 'condition'],
    'image': ['image_name', 'img_name', 'image_file', 'image_filename', 'image_path', 'image', 'filename', 'file_name', 'name'],
    'mask': ['label_file', 'label_name', 'label_filename', 'label_path', 'label', 'mask_file', 'mask_name', 'mask_filename', 'mask_path', 'mask', 'annotation'],
    'ratio': ['mask_ratio', 'lesion_ratio', 'ratio', 'mask_fraction'],
    'split': ['split', 'subset', 'partition', 'set', 'data_split', 'train_test_split']}
lookup = {norm(c): c for c in df.columns}
cols = {k: COLUMN_OVERRIDES.get(k, next((lookup[norm(a)] for a in v if norm(a) in lookup), None))
        for k, v in aliases.items()}
print('Detected columns (override in section 1):', cols)
for k in ('plant', 'disease', 'image', 'split'):
    if cols[k] not in df.columns:
        raise ValueError(f'Missing {k}: set COLUMN_OVERRIDES; official split is required.')
for k in ('mask', 'ratio'):
    if cols[k] is not None and cols[k] not in df.columns:
        raise ValueError(f'Invalid {k} override: {cols[k]}')
COL_PLANT, COL_DIS, COL_NAME, COL_LABEL, COL_RATIO, COL_SPLIT = [cols[k] for k in aliases]

# Index stems separately. Multiple matches are resolved by metadata path or official split,
# never by an arbitrary first file.
img_idx, mask_idx = defaultdict(list), defaultdict(list)
for p in DATA_DIR.rglob('*'):
    if not p.is_file() or p.suffix.lower() not in ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'):
        continue
    dirs = [x.lower() for x in p.relative_to(DATA_DIR).parts[:-1]]
    is_mask = any(any(k in part for k in ('mask', 'label', 'annotation')) for part in dirs)
    is_mask = is_mask or p.stem.lower().endswith(('_mask', '_label'))
    is_image = any('image' in part or part in ('img','imgs','rgb','photos') for part in dirs)
    if is_mask:
        mask_idx[p.stem.casefold()].append(p)
    elif is_image:
        img_idx[p.stem.casefold()].append(p)
print('Indexed image / mask stems:', len(img_idx), len(mask_idx))

def resolve(value, index, split, mask=False):
    if pd.isna(value):
        return None
    name = str(value).strip().replace('\\', '/')
    stem = Path(name).stem.casefold()
    keys = [stem] + ([stem + '_mask', stem + '_label'] if mask else [])
    candidates = list(dict.fromkeys(p for k in keys for p in index.get(k, [])))
    if len(candidates) > 1 and '/' in name:
        matches = [p for p in candidates if p.as_posix().casefold().endswith('/' + name.casefold().lstrip('/'))]
        if matches:
            candidates = matches
    if len(candidates) > 1:
        matches = [p for p in candidates if split in [x.lower() for x in p.parts]]
        if matches:
            candidates = matches
    return str(candidates[0]) if len(candidates) == 1 else None

split_values = df[COL_SPLIT].astype(str).str.strip().str.lower()
print('Official split values:', split_values.value_counts().to_dict())
df['official_split'] = split_values.replace({'training': 'train', 'testing': 'test'})
valid = df['official_split'].isin(['train', 'test'])
print('Rows outside official train/test (excluded):', int((~valid).sum()))
df = df.loc[valid].dropna(subset=[COL_PLANT, COL_DIS, COL_NAME]).copy()
df['img_path'] = [resolve(v, img_idx, s) for v, s in zip(df[COL_NAME], df.official_split)]
mask_names = df[COL_LABEL].fillna(df[COL_NAME]) if COL_LABEL else df[COL_NAME]
df['mask_path'] = [resolve(v, mask_idx, s, True) for v, s in zip(mask_names, df.official_split)]
print('Missing or ambiguous image / mask:', df.img_path.isna().sum(), df.mask_path.isna().sum())
df = df.dropna(subset=['img_path', 'mask_path']).copy()
# Remove conflicting image assignments, and repeated rows for the same physical file.
conflict = df.groupby('img_path')[[COL_PLANT, COL_DIS, 'official_split', 'mask_path']].nunique().max(axis=1)
df = df[~df.img_path.isin(conflict[conflict > 1].index)].drop_duplicates('img_path').reset_index(drop=True)
if df.empty or set(df.official_split) != {'train', 'test'}:
    raise ValueError('No usable official train/test pairs. Inspect paths and metadata mapping.')
print('Usable rows:', len(df))

## 4. Select crops and classes, then explore the data
Crop selection uses official training counts only to avoid test-driven cohort selection. The test split is retained for evaluation.

In [ ]:
df['crop'] = df[COL_PLANT].astype(str).str.strip()
df['disease'] = df[COL_DIS].astype(str).str.strip()
df = df[(df.crop != '') & (df.disease != '')].copy()
assert not df.crop.str.contains('__', regex=False).any(), 'Crop names must not contain __.'
df['label'] = df['crop'] + '__' + df['disease']
# Fit cohort selection only on official training data; test labels do not determine ranking.
training = df[df.official_split == 'train']
counts = training.label.value_counts()
good = counts[counts >= MIN_PER_CLASS].index
eligible = training[training.label.isin(good)]
top_crops = eligible.crop.value_counts().head(N_CROPS).index.tolist()
data = df[df.label.isin(good) & df.crop.isin(top_crops)].copy().reset_index(drop=True)
if data.empty:
    raise ValueError('No eligible crop-disease classes; check MIN_PER_CLASS and metadata.')
print('Selected crops:', top_crops, '| classes:', data.label.nunique(), '| images:', len(data))
if len(top_crops) < N_CROPS:
    print('Warning: fewer than the requested number of crops qualified.')
vc = data.label.value_counts().sort_values()
plt.figure(figsize=(9, max(3, 0.28 * len(vc))))
vc.plot.barh(); plt.title('Images per selected crop-disease class'); plt.xlabel('Images')
plt.tight_layout(); plt.show()
display(data.groupby('crop').agg(images=('label','size'), classes=('label','nunique')))

## 5. Severity labels from lesion masks
Lesion fraction = non-zero lesion pixels / all mask pixels. Mild < 0.05; moderate >= 0.05 and < 0.20; severe >= 0.20. These are fixed project thresholds, not clinical or expert grades; background is included in the denominator. Check mask encoding and metadata ratio units.

In [ ]:
def lesion_fraction(p):
    with Image.open(p) as image:
        m = np.asarray(image)
        if image.mode == "RGBA":
            m = m[..., :3]  # alpha is not a lesion channel
    lesion = (m != 0).any(axis=-1) if m.ndim == 3 else (m != 0)
    return float(lesion.mean())

data['lesion_frac'] = [lesion_fraction(p) for p in tqdm(data.mask_path)]
if COL_RATIO:
    raw = data[COL_RATIO].astype(str).str.strip()
    ratio = pd.to_numeric(raw.str.rstrip('%'), errors='coerce')
    if raw.str.endswith('%').any():
        ratio = ratio.where(~raw.str.endswith('%'), ratio / 100)
    elif ratio.max() > 1:
        print('Metadata ratio appears to be a percentage; dividing by 100. Verify units.')
        ratio = ratio / 100
    valid = ratio.between(0, 1)
    if valid.any():
        print('Mask ratio MAE:', (data.loc[valid, 'lesion_frac'] - ratio[valid]).abs().mean())
        print('Mask ratio correlation:', data.loc[valid, 'lesion_frac'].corr(ratio[valid]))
    else:
        print('No valid metadata ratios to compare.')

SEV_NAMES = ['mild', 'moderate', 'severe']
def to_grade(f):
    return 0 if f < SEV_T1 else (1 if f < SEV_T2 else 2)
data['sev'] = data.lesion_frac.map(to_grade)

plt.figure(figsize=(7, 3.5))
plt.hist(data.lesion_frac, bins=50)
plt.axvline(SEV_T1, color='r', ls='--'); plt.axvline(SEV_T2, color='r', ls='--')
plt.title('Lesion fraction distribution with severity thresholds'); plt.xlabel('lesion fraction'); plt.show()
print(data.sev.map(dict(enumerate(SEV_NAMES))).value_counts())

# examples: image, mask and grade
samp = data.sample(min(3, len(data)), random_state=1)
fig, ax = plt.subplots(2, 3, figsize=(11, 6))
for i, (_, r) in enumerate(samp.iterrows()):
    ax[0, i].imshow(Image.open(r.img_path).convert('RGB')); ax[0, i].axis('off')
    ax[0, i].set_title(r.label, fontsize=8)
    ax[1, i].imshow(Image.open(r.mask_path)); ax[1, i].axis('off')
    ax[1, i].set_title(f'lesion {r.lesion_frac:.1%} -> {SEV_NAMES[r.sev]}', fontsize=9)
plt.tight_layout(); plt.show()

## 6. Train / validation / test split, augmentation and data loaders

In [ ]:
classes = sorted(data.label.unique())
cls2idx = {c: i for i, c in enumerate(classes)}
data['y'] = data.label.map(cls2idx)

train_all = data[data.official_split == 'train'].copy()
test_df = data[data.official_split == 'test'].copy()
if train_all.empty or test_df.empty:
    raise ValueError('Selected cohort must have official train and test rows.')
try:
    train_df, val_df = train_test_split(train_all, test_size=0.15, stratify=train_all.y, random_state=SEED)
except ValueError:
    print('Stratification unavailable; using seeded random validation split.')
    train_df, val_df = train_test_split(train_all, test_size=0.15, random_state=SEED)
assert set(train_df.y) == set(range(len(classes))), 'Validation split removed a class from training.'
assert set(train_df.img_path).isdisjoint(val_df.img_path) and set(train_all.img_path).isdisjoint(test_df.img_path)
print('Train:', len(train_df), '| validation:', len(val_df), '| test:', len(test_df))

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class LeafDS(Dataset):
    def __init__(self, frame, tf):
        self.f, self.tf = frame.reset_index(drop=True), tf
    def __len__(self):
        return len(self.f)
    def __getitem__(self, i):
        r = self.f.iloc[i]
        img = Image.open(r.img_path).convert('RGB')
        return self.tf(img), int(r.y), int(r.sev)

train_dl = DataLoader(LeafDS(train_df, train_tf), batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
val_dl = DataLoader(LeafDS(val_df, eval_tf), batch_size=BATCH, num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
test_dl = DataLoader(LeafDS(test_df, eval_tf), batch_size=BATCH, num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))

## 7. Model: shared MobileNetV3-Small backbone with two heads

In [ ]:
class MultiTaskNet(nn.Module):
    def __init__(self, n_cls, n_sev=3):
        super().__init__()
        base = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT if PRETRAINED else None)
        self.backbone = base.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        def head(n):
            return nn.Sequential(nn.Linear(576, 256), nn.Hardswish(), nn.Dropout(0.2), nn.Linear(256, n))
        self.cls_head, self.sev_head = head(n_cls), head(n_sev)
    def forward(self, x):
        f = self.pool(self.backbone(x)).flatten(1)
        return self.cls_head(f), self.sev_head(f)

model = MultiTaskNet(len(classes)).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print('Parameters:', n_params, '| classes:', len(classes))

## 8. Training
AdamW, equal-weight cross-entropy losses. Retain the checkpoint with minimum combined validation loss; do not select using test metrics.

In [ ]:
ce = nn.CrossEntropyLoss()
opt = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 1e-4},
    {'params': list(model.cls_head.parameters()) + list(model.sev_head.parameters()), 'lr': 1e-3}])

def run_epoch(loader, train):
    model.train(train)
    tl = c1 = c2 = n = 0
    for x, y, s in loader:
        x, y, s = x.to(DEVICE), y.to(DEVICE), s.to(DEVICE)
        with torch.set_grad_enabled(train):
            o1, o2 = model(x)
            loss = ce(o1, y) + ce(o2, s)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
        tl += loss.item() * len(x); n += len(x)
        c1 += (o1.argmax(1) == y).sum().item(); c2 += (o2.argmax(1) == s).sum().item()
    return tl / n, c1 / n, c2 / n

hist = []
best, best_state = float("inf"), None
best_epoch = None
for ep in range(1, EPOCHS + 1):
    tr = run_epoch(train_dl, True)
    va = run_epoch(val_dl, False)
    hist.append((*tr, *va))
    if va[0] < best:
        best, best_epoch = va[0], ep
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f'Epoch {ep:02d} | train loss {tr[0]:.3f} cls {tr[1]:.3f} sev {tr[2]:.3f} | val loss {va[0]:.3f} cls {va[1]:.3f} sev {va[2]:.3f}')
assert best_state is not None, "Training did not produce a valid checkpoint."
model.load_state_dict(best_state)
print("Best validation-loss epoch:", best_epoch)

h = np.array(hist)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(range(1, len(h)+1), h[:, 0], label='train'); ax[0].plot(range(1, len(h)+1), h[:, 3], label='val'); ax[0].set_title('Loss'); ax[0].legend()
ax[1].plot(range(1, len(h)+1), h[:, 1], label='train disease'); ax[1].plot(range(1, len(h)+1), h[:, 4], label='val disease')
ax[1].plot(range(1, len(h)+1), h[:, 2], '--', label='train severity'); ax[1].plot(range(1, len(h)+1), h[:, 5], '--', label='val severity')
ax[1].set_title('Accuracy'); ax[1].legend(); plt.tight_layout(); plt.show()

## 9. Evaluation on the held-out test set

In [ ]:
model.eval()
P1, P2, T1, T2 = [], [], [], []
with torch.no_grad():
    for x, y, s in test_dl:
        o1, o2 = model(x.to(DEVICE))
        P1 += o1.argmax(1).cpu().tolist(); P2 += o2.argmax(1).cpu().tolist()
        T1 += y.tolist(); T2 += s.tolist()
P1, P2, T1, T2 = map(np.array, (P1, P2, T1, T2))

print('Disease (crop + disease) accuracy:', round(accuracy_score(T1, P1), 4))
crop_of = lambda i: classes[i].split('__')[0]
print('Crop-level accuracy:', round(np.mean([crop_of(a) == crop_of(b) for a, b in zip(T1, P1)]), 4))
print('Severity accuracy:', round(accuracy_score(T2, P2), 4))
print('Disease macro F1:', classification_report(T1, P1, labels=range(len(classes)), output_dict=True, zero_division=0)['macro avg']['f1-score'])
print('Severity macro F1:', classification_report(T2, P2, labels=range(3), output_dict=True, zero_division=0)['macro avg']['f1-score'])
print('\nDisease classification report:')
print(classification_report(T1, P1, labels=range(len(classes)), target_names=classes, zero_division=0))
print('Severity classification report:')
print(classification_report(T2, P2, labels=[0, 1, 2], target_names=SEV_NAMES, zero_division=0))

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
cm = confusion_matrix(T1, P1, labels=range(len(classes)), normalize='true')
ax[0].imshow(cm, cmap='Blues'); ax[0].set_title('Disease confusion matrix (row-normalised)')
if len(classes) <= 40:
    ax[0].set_xticks(range(len(classes))); ax[0].set_yticks(range(len(classes)))
    ax[0].set_xticklabels(classes, rotation=90, fontsize=6); ax[0].set_yticklabels(classes, fontsize=6)
cm2 = confusion_matrix(T2, P2, labels=[0, 1, 2])
ax[1].imshow(cm2, cmap='Blues'); ax[1].set_title('Severity confusion matrix')
ax[1].set_xticks(range(3)); ax[1].set_yticks(range(3)); ax[1].set_xticklabels(SEV_NAMES); ax[1].set_yticklabels(SEV_NAMES)
for i in range(3):
    for j in range(3):
        ax[1].text(j, i, cm2[i, j], ha='center', va='center')
for a in ax:
    a.set_xlabel('Predicted'); a.set_ylabel('True')
plt.tight_layout(); plt.show()

## 10. Export for offline use

In [ ]:
model = model.eval().cpu()
example = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
with torch.inference_mode():
    traced = torch.jit.trace(model, example)
    for a, b in zip(model(example), traced(example)):
        torch.testing.assert_close(a, b)
traced.save('plant_model.pt')
meta = {'classes': classes, 'severity_names': SEV_NAMES, 'thresholds': [SEV_T1, SEV_T2],
        'img_size': IMG_SIZE, 'mean': MEAN, 'std': STD}
with open('plant_model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
for path in ('plant_model.pt', 'plant_model_meta.json'):
    print(path, os.path.getsize(path), 'bytes')
try:
    torch.onnx.export(model, example, 'plant_model.onnx', input_names=['image'],
                      output_names=['disease', 'severity'], opset_version=17, dynamo=False)
    print('plant_model.onnx:', os.path.getsize('plant_model.onnx'), 'bytes')
except Exception as e:
    print('Optional ONNX export skipped:', e)

## 11. Offline inference demo
This cell uses only the exported file and the metadata JSON, with no training code and no internet.

In [ ]:
# Standalone inference: run these imports and definitions in a fresh offline Python session.
import json, time
import torch
from PIL import Image
from torchvision import transforms

loaded = torch.jit.load('plant_model.pt', map_location='cpu').eval()
with open('plant_model_meta.json') as f:
    meta = json.load(f)
preprocess = transforms.Compose([transforms.Resize((meta['img_size'], meta['img_size'])),
    transforms.ToTensor(), transforms.Normalize(meta['mean'], meta['std'])])
# Warm up CPU model before timing.
with torch.inference_mode():
    loaded(torch.zeros(1, 3, meta['img_size'], meta['img_size']))

def predict(path):
    t0 = time.perf_counter()
    with Image.open(path) as image:
        x = preprocess(image.convert('RGB')).unsqueeze(0)
    with torch.inference_mode():
        d, s = loaded(x)
        p, q = d.softmax(1)[0], s.softmax(1)[0]
    i, j = int(p.argmax()), int(q.argmax())
    crop, disease = meta['classes'][i].split('__', 1)
    return dict(crop=crop, disease=disease, confidence=float(p[i]),
                severity=meta['severity_names'][j], severity_confidence=float(q[j]),
                milliseconds=(time.perf_counter()-t0)*1000)

# Evaluation display uses held-out paths only; predict itself needs no training objects.
demo = test_df.sample(min(4, len(test_df)), random_state=3)
fig, axes = plt.subplots(1, len(demo), figsize=(4*len(demo), 4.5), squeeze=False)
times = []
for a, (_, r) in zip(axes[0], demo.iterrows()):
    out = predict(r.img_path)
    times.append(out['milliseconds'])
    a.imshow(Image.open(r.img_path).convert('RGB')); a.axis('off')
    a.set_title(f"True: {r.label} / {meta['severity_names'][r.sev]}\nPred: {out['crop']} / {out['disease']}\n"
                f"Confidence: {out['confidence']:.1%} | {out['severity']}\n{out['milliseconds']:.1f} ms", fontsize=8)
plt.tight_layout(); plt.show()
print('Mean CPU end-to-end time (demo images, after warmup):', sum(times)/len(times), 'ms')
# print(predict('/content/my_leaf.jpg'))

## 12. Download the exported model (backup for the viva)

In [ ]:
from google.colab import files
files.download('plant_model.pt')
files.download('plant_model_meta.json')